# Workshop 4 (CBA)

In [ ]:
import os

## Workflow management using `Snakemake` and `pixi`

- Talk about how the Open-TYNDP project utilizes `Snakemake` as a workflow management system
- We also use `pixi` to manage environments and to enable ease-of-use of `Snakemake`

### Reminder: `Snakemake`

<img src="https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/792b8474ab5096e5ab8db2822af4fcd9fe659eb6/open-tyndp-workshops/snakemake_logo.png" width="500px" />

The `Snakemake` workflow management system is a tool to create reproducible and scalable data analyses.
Workflows are described via a human readable, Python based language. They can be seamlessly scaled to server, cluster, grid, and cloud environments, without the need to modify the workflow definition.

Snakemake follows the [GNU Make](https://www.gnu.org/software/make) paradigm: workflows are defined in terms of so-called `rules` that specify how to create a set of output files from a set of input files. Dependencies between the rules are determined automatically, creating a DAG (directed acyclic graph) of jobs that can be automatically parallelized.

:::{note}
Documentation for this package is available at https://snakemake.readthedocs.io/. You can also check out a [slide deck Snakemake Tutorial](https://slides.com/johanneskoester/snakemake-tutorial) by Johannes Köster (2024).

Mölder, F., Jablonski, K.P., Letcher, B., Hall, M.B., Tomkins-Tinch, C.H., Sochat, V., Forster, J., Lee, S., Twardziok, S.O., Kanitz, A., Wilm, A., Holtgrewe, M., Rahmann, S., Nahnsen, S., Köster, J., 2021. Sustainable data analysis with Snakemake. F1000Res 10, 33.

Add note to refer to previous workshop
:::


#### Using `snakemake`

You can trigger the workflow by specifying a target file, like `data/benchmark.pdf`, or any intermediate file:
```bash
snakemake -call data/benchmark.pdf
```

Alternatively, you can also execute the workflow by calling a rule that produces an intermediate file:
```bash
snakemake -call build_data
```
**NOTE:** You cannot call a rule that includes a wildcard without specifying what the wildcard should be filled with. Otherwise, Snakemake will not know what to propagate back.

Or you can call the common rule `all` which can be used to execute the entire workflow. It takes the final workflow output as its input and thus requires all previous dependent rules to be run as well:
```bash
snakemake -call all
```

Because we defined the `all` rule as first in the `Snakefile`, this rule is assumed to be the default and the following also works:
```bash
snakemake -call
```

A very important feature is the `-n` flag which executes a `dry-run`. It is recommended to always first execute a `dry-run` before the actual execution of a workflow. This simply prints out the DAG of the workflow to investigate without actually executing it.

Let's try this out and investigate the output:

In [ ]:
! snakemake -call -n

### Introducing: `pixi`

<img src="https://raw.githubusercontent.com/prefix-dev/pixi/9f94b8a24353ca88d21846b72438c6066e8adc74/docs/assets/pixi-logo.svg" width="150px" />



`pixi` is a cross-platform, multi-language package management and workflow tool. It is built on the foundation of the `conda` ecosystem.

:::{note}
Documentation for this package is available at https://pixi.prefix.dev/latest/. 

:::


#### Using `pixi`

- we are using `pixi` for package/environment management
- we are able to wrap snakemake commands into pixi commands
- for remainder of notebook, we will use pixi

Give examples of how we wrapped specific snakemake commands into pixi commands, to allow us to run shorter commands

### Coupled vs decoupled SB-CBA workflow

<center>

![](cba_multi_proj.png)
![](cba_from_presolved.png)

</center>

Note:
- Connection between SB solve and CBA
- Mention that there are many many rules before solve_sector_network_myopic - we just chose to highlight the handoff between SB and CBA here
- Number of projects - only 2 shown here but for each project, the DAG expands and blows up

Note:
- With this there are no rules before the retrieve rule - we start from this point
- But all else is the same

## Exploring Open-TYNDP CBA

In [ ]:
# Uncomment the next line for running this notebook on Colab
# !wget -qO- https://pixi.sh/install.sh | sh

### Cloning the Open-TYNDP project

First, navigate into the `data/` folder:

In [ ]:
# TODO: create data/ folder if it doesn't exist, and cd into it before cloning the repo
os.chdir("data/")
print("Directory changed to:", os.getcwd())

Clone the Open-TYNDP repository directly:

In [ ]:
if os.path.basename(os.getcwd()) == "data":
    if not os.path.exists("open-tyndp"):
        ! git clone https://github.com/open-energy-transition/open-tyndp.git
    else:
        print("open-tyndp directory already exists, skipping clone")
else:
    print("Warning: Not in data/ directory. Current directory:", os.getcwd())

Now, navigate into the the `open-tyndp` directory:

In [ ]:
os.chdir("open-tyndp/")
print("Directory changed to:", os.getcwd())

In [ ]:
! git checkout cba-low-res

(add instruction to install pixi - copy from workshop 3)

Use `pixi` to install the `open-tyndp` environment:

In [ ]:
os.environ["PATH"] = os.path.expanduser("~/.pixi/bin") + ":" + os.environ["PATH"]

In [ ]:
! pixi install -e open-tyndp

### Running a coupled SB->CBA workflow

To run the CBA workflow, the `pixi run tyndp-cba` runs the full process: taking a solved SB network -> performing the TOOT/PINT analysis on a project -> calculating the CBA indicators. 

The default Open-TYNDP settings can be found in the `config/config.tyndp.yaml` file. 

TODO: add chunks of config file (refer to previous workshop about manual adjustments)
- add snippets of default values (runs, horizons, snapshots, projects, etc)

mention:
- because of notebook environment, we are focusing on command line changes, but in reality it's easier to modify in config files directly using dedicated IDE

First, let's check what running the full could look like by doing a dry-run of `pixi run tyndp-cba`:

In [ ]:
! pixi run tyndp-cba -n

We can specify the specific scenario (e.g, `NT`), the project (e.g., `t4`), and planning horizon (e.g., 2030) we want to run also directly in the command line. 

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

Notice how the `count` of steps changed when we specified a single run, project, and horizon. 
Comparatively, the default settings run the full list of scenarios we have, 5 projects (t4, t16, t28, t33, t35), and two planning horizons (2030 and 2040). So, the full default workflow will run many more steps.

#### Checkpoints

You may notice above that the number of rules is actually quite low.

There is a `checkpoint` in the workflow, called `clean_projects`, that first checks how many projects are being asked to run before building out the full DAG. 
It tells the workflow which CBA projects exist, which project IDs to run, and which method applies (TOOT/PINT). 

Before the `clean_projects` step runs, Snakemake does not know the full list of project jobs it needs to create. 
The step downloads and cleans the external CBA project database, tells the workflow the full list of projects available, which projects the user wants to evaluate, and how each should be evaluated.
After `clean_projects` finishes, Snakemake can read the cleaned CSV and expand the DAG into concrete jobs.

Thus, what we should do first here is run the workflow just up until the checkpoint, then only after that can we run the full workflow again -- in which case, the DAG would show the actual number of jobs that would be run.

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

Now that the checkpoint is complete, we can re-check the DAG of how many steps are needed to run the CBA workflow.

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

#### Task X: Change the configuration directly in the `config/config.tyndp.yaml`

- Change the config settings to match above
  - Change scenario name in config.tyndp.yaml
  - Change parameters within NT in scenarios.tyndp.yaml
- Come back here
- Run simplified command line

In [ ]:
# Should give the same output as above, without the arguments
! pixi run tyndp-cba -n

### Running different and multiple climate years

(Add some info here about how different climate scenarios are defined? Navigate into `config/scenarios.yaml`?)

TODO:
- add snippets of yaml here to show as examples
- can also show live
- stress that in practice it's easier to modify in yaml

First, we again need to run up to the checkpoint for the different climate years:

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":["NT-cy2008", "NT-cy2009", "NT-cy1995"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

So, we can run for just another climate year, such as the 1995 climate year for the NT scenario (`NT-cy2008`):

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["NT-cy2008"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

Or, we can run all climate years (1995, 2008, and 2009) for the `NT` scenario:

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["NT-cyears"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

Note how the total count of steps is much higher (especially for CBA-specific steps such as `make_indicators` and weather-dependent steps such as `build_renewable_profiles_pecd`) when multiple climate years are run.

#### Task X: Create your own climate year and run it

Go into `scenarios.tyndp.yaml` to create your own climate year run.

Hints:
1. Copy an existing climate year run (e.g., `NT-cy2008`) and modify the details to your liking. Change the name.
2. You would need to first run the new climate year up until the checkpoint.
3. After the checkpoint, you can use `pixi run tyndp-cba` to run the full workflow for your new climate year. 

Warning:
We used previous set of PECD data. So we can only run 1995, 2008, and 2009 at the moment.

Mention the benefit: if we have more climate years in the year, this is how we would add it.

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":["frog"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["frog"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

### Running a decoupled SB->CBA Workflow

Based on the DAGs shown in our dry runs, you can see that there are quite a lot of steps and rules that need to be run. A lot of these rules are specific to the SB stage of Open-TYNDP. 

We have the flexibility to bypass running and solving the SB workflow each time, by instead downloading a pre-solved SB network that we have released. By doing this, when running the CBA workflow, we start the process from an already solved SB network, reducing the number of steps significantly.

This can be easily done by setting the `cba.use_presolved` setting to `true`. By default, this downloads and uses the latest release we have (at the time of this workshop, that is v0.7.1).

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["frog"]}' 'cba={"cba_scenario_input":{"use_presolved":true,"sb_version":"latest"},"planning_horizons":[2030],"projects":["t4"]}' -n

Note how the number of steps has decreased down from over 100 to just ~37, and we see some new steps not previously seen before, such as `retrieve_presolved_sb_networks`. 

#### Task (optional): Running and solving a CBA project with a pre-solved network

- Checkout to workshop branch
- Mention warning that it takes 15 minutes to solve
- Let people run it during the break